# Second part : Classifying the preprocessed dataset using a single Decision Tree then a Random Forest with various Hyperparameters


In [18]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold

In [19]:
# read dataset
X_test = pd.read_csv('../preprocessed dataset\X_test.csv')
X_train = pd.read_csv('../preprocessed dataset\X_train.csv')
y_train = pd.read_csv('../preprocessed dataset\y_train.csv')

# Convert y_train to 1D array (from DataFrame)
y_train = y_train.values.ravel()

In [20]:
# Keep TransactionID for later use in submission file
test_ids = X_test['TransactionID'].copy()
# Drop TransactionID from both sets for training
X_train = X_train.drop(columns=['TransactionID'], errors='ignore')
X_test = X_test.drop(columns=['TransactionID'], errors='ignore')

In [21]:
# Split training data into train and validation sets (80% train, 20% validation)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)


In [22]:
dt_model = DecisionTreeClassifier(random_state=1)
dt_model.fit(X_tr, y_tr)
dt_pred = dt_model.predict(X_val)
dt_proba = dt_model.predict_proba(X_val)[:, 1]

print("=== Decision Tree (on validation set) ===")
print("Accuracy:", accuracy_score(y_val, dt_pred))
print("F1 Score:", f1_score(y_val, dt_pred))
print("ROC AUC:", roc_auc_score(y_val, dt_proba))
print("Confusion Matrix:\n", confusion_matrix(y_val, dt_pred))
print(classification_report(y_val, dt_pred, target_names=['Not Fraud', 'Fraud']))

=== Decision Tree (on validation set) ===
Accuracy: 0.7955480280667796
F1 Score: 0.7994779307071666
ROC AUC: 0.7955480280667796
Confusion Matrix:
 [[3207  926]
 [ 764 3369]]
              precision    recall  f1-score   support

   Not Fraud       0.81      0.78      0.79      4133
       Fraud       0.78      0.82      0.80      4133

    accuracy                           0.80      8266
   macro avg       0.80      0.80      0.80      8266
weighted avg       0.80      0.80      0.80      8266



## Random Forest Classifier

In [23]:
# Basic Random Forest with default parameters (for comparison)
rf_default = RandomForestClassifier(criterion='entropy', random_state=50, n_estimators=100)
rf_default.fit(X_tr, y_tr)
rf_pred = rf_default.predict(X_val)
rf_proba = rf_default.predict_proba(X_val)[:, 1]

print("\n=== Random Forest (default, on validation set) ===")
print("Accuracy:", accuracy_score(y_val, rf_pred))
print("F1 Score:", f1_score(y_val, rf_pred))
print("ROC AUC:", roc_auc_score(y_val, rf_proba))
print("Confusion Matrix:\n", confusion_matrix(y_val, rf_pred))
print(classification_report(y_val, rf_pred, target_names=['Not Fraud', 'Fraud']))



=== Random Forest (default, on validation set) ===
Accuracy: 0.8518025647229616
F1 Score: 0.8495270851246776
ROC AUC: 0.9302395682300503
Confusion Matrix:
 [[3583  550]
 [ 675 3458]]
              precision    recall  f1-score   support

   Not Fraud       0.84      0.87      0.85      4133
       Fraud       0.86      0.84      0.85      4133

    accuracy                           0.85      8266
   macro avg       0.85      0.85      0.85      8266
weighted avg       0.85      0.85      0.85      8266



In [28]:
# Hyperparameter tuning with GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 400],
    'max_depth': [None, 10, 20, 40],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 5]
}

rf_grid = RandomForestClassifier(random_state=50, criterion='entropy')
grid_search = GridSearchCV(rf_grid, param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=0)
grid_search.fit(X_tr, y_tr)

# Display all results
results_df = pd.DataFrame(grid_search.cv_results_)

# Show relevant columns: parameters, mean test score, std test score
display_cols = ['params', 'mean_test_score', 'std_test_score']
results_df = results_df[display_cols].sort_values('mean_test_score', ascending=False)
print("\n=== All Grid Search Results ===")
print(results_df.to_string(index=False))

# Define StratifiedKFold for cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# After grid search, you can also evaluate the best model with CV:
best_rf = grid_search.best_estimator_
best_cv_scores = cross_val_score(best_rf, X_train, y_train, cv=cv, scoring='roc_auc')
print("\n=== Best Random Forest (5‑fold CV) ===")
print("ROC AUC scores:", best_cv_scores)
print("Mean ROC AUC: {:.4f} ± {:.4f}".format(best_cv_scores.mean(), best_cv_scores.std()))


=== All Grid Search Results ===
                                                                                  params  mean_test_score  std_test_score
   {'max_depth': 40, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 400}         0.921813        0.003745
 {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 400}         0.921789        0.004168
   {'max_depth': 40, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}         0.921156        0.003762
 {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}         0.920917        0.004051
 {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 400}         0.920545        0.003923
   {'max_depth': 40, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 400}         0.920543        0.004182
   {'max_depth': 40, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}         0.920021        

In [29]:
# Train the best model (here we use the default random forest) on the full training data
final_model = grid_search.best_estimator_
final_model.fit(X_train, y_train)



# Predict probabilities on the test set
test_proba = final_model.predict_proba(X_test)[:, 1]

# Create submission DataFrame
if test_ids is not None:
    submission = pd.DataFrame({'TransactionID': test_ids, 'isFraud': test_proba})
else:
    # If no TransactionID, create index as ID (less ideal)
    submission = pd.DataFrame({'isFraud': test_proba})
    submission.index.name = 'TransactionID'

submission.to_csv('submission.csv', index=False)
print("\nSubmission file saved as submission.csv")


Submission file saved as submission.csv
